## The Noise groups

In [21]:
from astropy.io import fits
import numpy as np
from astropy.table import Table
import pandas as pd
import glob
from astropy.table import vstack
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages

In [22]:
def error_cut(df, jhk_thresh=0.2, wise_thresh=0.5):
    """
    Filtro de calidad ajustable para estudios de SySts
    
    Args:
        df: DataFrame de entrada
        jhk_thresh: Umbral para J,H,K (default 0.2)
        wise_thresh: Umbral para bandas WISE (default 0.5)
    
    Returns:
        DataFrame filtrado
    """
    good_quality = (
        (df['e_Jmag'] < jhk_thresh) &
        (df['e_Hmag'] < jhk_thresh) &
        (df['e_Kmag'] < jhk_thresh) &
        (df['e_W1mag'] < wise_thresh) &
        (df['e_W2mag'] < wise_thresh) &
        (df['e_W3mag'] < wise_thresh) &
        (df['e_W4mag'] < wise_thresh)
    )
    return df[good_quality]

In [23]:
def colours(df):
    """
    Calcula TODOS los colores necesarios para los criterios de Akras
    """
    # Colores existentes
    df['H_W2'] = df['Hmag'] - df['W2mag']
    df['Ks_W3'] = df['Kmag'] - df['W3mag']
    df['J_H'] = df['Jmag'] - df['Hmag']
    df['W1_W2'] = df['W1mag'] - df['W2mag']
    df['W1_W4'] = df['W1mag'] - df['W4mag']
    
    # Nuevos colores necesarios para clasificación
    df['W3_W4'] = df['W3mag'] - df['W4mag']  # Necesario para S+IR y D
    # df['H_W2'] ya existe, no es necesario duplicar
    
    return df

In [24]:
def akras_criterion_ii(df):
    """
    Aplica el criterio IR (ii) de Akras et al. (2019b) para seleccionar candidatos a 
    estrellas simbióticas de tipo S (S-type SySts).

    Criterio (ii):
        (1) J - H ≥ 0.78 
            AND 0 < Ks - W3 < 1.18 
            AND W1 - W2 < 0.09
        OR
        (2) J - H ≥ 0.78 
            AND 0 < Ks - W3 < 1.18 
            AND W1 - W2 ≥ 0.09 
            AND 0 < W1 - W4 < 0.92

    Args:
        df (pd.DataFrame): DataFrame con las columnas de colores calculadas.

    Returns:
        pd.Series: Máscara booleana (True para fuentes que cumplen el criterio).
    """
    # Condiciones comunes a ambos sub-criterios
    base_condition = (df['J_H'] >= 0.78) & (df['Ks_W3'] > 0) & (df['Ks_W3'] < 1.18)
    
    # Sub-criterio 1
    condition1 = base_condition & (df['W1_W2'] < 0.09)
    
    # Sub-criterio 2
    condition2 = base_condition & (df['W1_W2'] >= 0.09) & (df['W1_W4'] > 0) & (df['W1_W4'] < 0.92)
    
    return condition1 | condition2

### Aplying

In [25]:
df_g4 = pd.read_csv("../Class_wise_v4/Halpha_emitter_wise_group4.csv")
len(df_g4)

301

In [76]:

df_g4_clean = error_cut(df_g4, wise_thresh=0.3)
len(df_g4_clean)

171

In [60]:
# Creating the cororls
df_g4_clean = colours(df_g4_clean)

/tmp/ipykernel_88192/154282448.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['H_W2'] = df['Hmag'] - df['W2mag']
/tmp/ipykernel_88192/154282448.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Ks_W3'] = df['Kmag'] - df['W3mag']
/tmp/ipykernel_88192/154282448.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/

In [61]:
# Applying the Akras criterio
mask_syst_candidates = akras_criterion_ii(df_g4_clean)
syst_candidates = df_g4_clean[mask_syst_candidates]
len(syst_candidates)

0

In [62]:
syst_candidates

,Name,RAJ2000,DEJ2000,GLON,GLAT,SourceID,ePos,Class,pStar,pGalaxy,...,PC3,PC4,PC5,Label,H_W2,Ks_W3,J_H,W1_W2,W1_W4,W3_W4


In [63]:
# 4. Limpieza básica de datos (opcional pero recomendado)
# - Eliminar objetos sin magnitudes críticas
syst_candidates = syst_candidates.dropna(subset=['Hmag', 'W2mag', 'Kmag', 'W3mag', 'Jmag', 'Hmag'])

In [64]:
# 5. Seleccionar columnas clave para crossmatch
cols_to_keep = ['Name', 'RAJ2000', 'DEJ2000', 'Jmag', 'Hmag', 'Kmag', 'W1mag', 'W2mag', 'W3mag', 'H_W2', 'Ks_W3', 'J_H']
syst_candidates = syst_candidates[cols_to_keep]

In [65]:
len(syst_candidates)

0

# Noise

In [66]:
df_noise= pd.read_csv("../Class_wise_v4/Halpha_emitter_wise_noise.csv")
len(df_noise)

1750

In [67]:
df_noise_clean = error_cut(df_noise, wise_thresh=0.3)
len(df_noise_clean)

948

In [68]:
# Creating the cororls
df_noise_clean = colours(df_noise_clean)

/tmp/ipykernel_88192/154282448.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['H_W2'] = df['Hmag'] - df['W2mag']
/tmp/ipykernel_88192/154282448.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Ks_W3'] = df['Kmag'] - df['W3mag']
/tmp/ipykernel_88192/154282448.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/

In [69]:
# Applying the Akras criterio
mask_syst_candidates_noise = akras_criterion_ii(df_noise_clean)
syst_candidates_noise = df_noise_clean[mask_syst_candidates_noise]
len(syst_candidates_noise)

4

In [70]:
syst_candidates_noise

,Name,RAJ2000,DEJ2000,GLON,GLAT,SourceID,ePos,Class,pStar,pGalaxy,...,PC3,PC4,PC5,Label,H_W2,Ks_W3,J_H,W1_W2,W1_W4,W3_W4
429,J214847.13+583226.7,327.196368,58.540741,100.966015,3.689364,420556-2-1274,0.045,99.0,1.0,0.0,...,5.505429,5.316182,4.837127,-1,0.482,0.761,0.823,-0.036,1.849,1.296
976,J183501.83+014655.8,278.757612,1.782156,32.649024,4.462744,568855-1-10595,0.053,-1.0,1.0,0.0,...,5.109161,3.483212,5.871586,-1,0.866,1.042,1.269,0.052,1.295,0.512
996,J185323.58+084955.1,283.348268,8.831966,41.024060,3.583785,464654-4-1631,0.060,-1.0,1.0,0.0,...,5.677661,3.818215,5.665704,-1,0.870,0.819,1.278,0.011,1.185,0.696
1460,J003024.37+652246.1,7.601532,65.379478,120.741721,2.597209,480211-1-3866,0.037,-1.0,1.0,0.0,...,5.448435,6.445526,4.309062,-1,0.461,1.017,0.832,0.082,4.996,4.093


In [71]:
# 9. Guardar para crossmatch con LAMOST
syst_candidates_noise.to_csv('../Class_wise_v4/SySt_S-type_candidates_Noise.csv', index=False)

# S + IR-type criterion

In [72]:
def akras_criterion_s_plus_ir(df):
    """
    Aplica el criterio S+IR-type de Akras et al. para clasificación de estrellas simbióticas.
    
    Criterio (iv)(2):
        (i) Ks - W3 < 1.93 and W3 - W4 ≥ 1.46
        OR
        (ii) Ks - W3 ≥ 1.93 and H - W2 < 2.72
    
    Requiere:
        - Ks_W3 = Kmag - W3mag
        - W3_W4 = W3mag - W4mag
        - H_W2 = Hmag - W2mag
    
    Args:
        df (pd.DataFrame): DataFrame con columnas de colores calculadas
        
    Returns:
        pd.Series: Máscara booleana (True para fuentes que cumplen el criterio S+IR-type)
    """
    # Primera condición: Ks_W3 < 1.93 and W3_W4 >= 1.46
    cond1 = (df['Ks_W3'] < 1.93) & (df['W3_W4'] >= 1.46)
    
    # Segunda condición: Ks_W3 >= 1.93 and H_W2 < 2.72
    cond2 = (df['Ks_W3'] >= 1.93) & (df['H_W2'] < 2.72)
    
    return cond1 | cond2

In [73]:
# Applying the Akras criterio S + IR
mask_syst_candidates_s_ir = akras_criterion_s_plus_ir(df_g4_clean)
syst_candidates_s_ir = df_g4_clean[mask_syst_candidates_s_ir]
len(syst_candidates_s_ir)

172

# Clasificación, todos los tipos

In [74]:
def classify_all_syst(df):
    """
    Clasifica todos los objetos en un DataFrame según los criterios de Akras para SySt
    
    Args:
        df: DataFrame con columnas necesarias para calcular los colores
        
    Returns:
        pd.Series con clasificación para cada objeto:
        - 'S-type', 'S+IR-type', 'D-type', "D'-type" para SySt confirmados
        - 'Not SySt' para objetos que no cumplen criterios
    """
    # Calcular colores necesarios si no existen
    if 'J_H' not in df.columns:
        df['J_H'] = df['Jmag'] - df['Hmag']
    if 'Ks_W3' not in df.columns:
        df['Ks_W3'] = df['Kmag'] - df['W3mag']
    if 'W1_W2' not in df.columns:
        df['W1_W2'] = df['W1mag'] - df['W2mag']
    if 'W1_W4' not in df.columns:
        df['W1_W4'] = df['W1mag'] - df['W4mag']
    if 'H_W2' not in df.columns:
        df['H_W2'] = df['Hmag'] - df['W2mag']
    if 'W3_W4' not in df.columns:
        df['W3_W4'] = df['W3mag'] - df['W4mag']
    
    # Inicializar resultados
    classifications = pd.Series('Not SySt', index=df.index)
    
    # 1. Criterio base para SySt (ii)
    base_cond = (df['J_H'] >= 0.78) & (df['Ks_W3'] > 0) & (df['Ks_W3'] < 1.18)
    cond1 = base_cond & (df['W1_W2'] < 0.09)
    cond2 = base_cond & (df['W1_W2'] >= 0.09) & (df['W1_W4'] > 0) & (df['W1_W4'] < 0.92)
    syst_mask = cond1 | cond2
    
    # 2. Criterio adicional (iii)
    syst_mask = syst_mask & (df['H_W2'] >= 0.206) & (df['Ks_W3'] >= 0.27)
    
    # 3. Clasificación de tipos (iv)
    # S-type
    s_mask = syst_mask & (df['Ks_W3'] < 1.93) & (df['W3_W4'] < 1.46)
    classifications[s_mask] = 'S-type'
    
    # S+IR-type
    s_ir_mask = syst_mask & (
        ((df['Ks_W3'] < 1.93) & (df['W3_W4'] >= 1.46)) | 
        ((df['Ks_W3'] >= 1.93) & (df['H_W2'] < 2.72)))
    classifications[s_ir_mask] = 'S+IR-type'
    
    # D-type
    d_mask = syst_mask & (df['Ks_W3'] >= 1.93) & (df['H_W2'] >= 2.72) & (df['W3_W4'] < 1.52)
    classifications[d_mask] = 'D-type'
    
    # D'-type
    d_prime_mask = syst_mask & (df['Ks_W3'] >= 1.93) & (df['H_W2'] >= 2.72) & (df['W3_W4'] >= 1.52)
    classifications[d_prime_mask] = "D'-type"
    
    return classifications

In [75]:
# Clasificar todos los objetos en df_g4_clean
df_g4_clean['syst_class'] = classify_all_syst(df_g4_clean)

# Ver distribución de clases
print("Distribución de clasificaciones SySt:")
print(df_g4_clean['syst_class'].value_counts())

# Filtrar solo objetos clasificados como SySt
syst_objects = df_g4_clean[df_g4_clean['syst_class'] != 'Not SySt']
print(f"\nTotal de objetos clasificados como SySt: {len(syst_objects)}")

# Mostrar detalles de los SySt
print(syst_objects[['J_H', 'Ks_W3', 'W1_W2', 'W1_W4', 'H_W2', 'W3_W4', 'syst_class']])

Distribución de clasificaciones SySt:
syst_class
Not SySt    215
Name: count, dtype: int64

Total de objetos clasificados como SySt: 0
Empty DataFrame
Columns: [J_H, Ks_W3, W1_W2, W1_W4, H_W2, W3_W4, syst_class]
Index: []


/tmp/ipykernel_88192/364637935.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_g4_clean['syst_class'] = classify_all_syst(df_g4_clean)


## Buscando objetos particulaes

In [82]:
def buscar_objeto_por_coordenadas(df, ra, dec, ra_col='RAJ2000', dec_col='DEJ2000'):
    """
    Busca el objeto más cercano a las coordenadas dadas (ra, dec) en el DataFrame,
    y retorna un nuevo DataFrame con esa fila.
    
    Parámetros:
        df (pd.DataFrame): El DataFrame donde buscar.
        ra (float): Coordenada RA a buscar.
        dec (float): Coordenada DEC a buscar.
        ra_col (str): Nombre de la columna RA en el DataFrame.
        dec_col (str): Nombre de la columna DEC en el DataFrame.
        
    Retorna:
        pd.DataFrame: DataFrame con la fila del objeto más cercano.
    """
    # Calcula la distancia angular simple (en grados)
    dist = np.sqrt((df[ra_col] - ra)**2 + (df[dec_col] - dec)**2)
    idx_min = dist.idxmin()  # Índice del objeto más cercano
    return df.loc[[idx_min]].copy()  # Retorna como DataFrame


In [83]:
Obj_exotic = buscar_objeto_por_coordenadas(df_g4_clean, 88.2253330, 17.24019400)

In [88]:
Obj_exotic[["rImag", "Hamag", "imag", 'W1mag',
 'W2mag',
 'W3mag',
 'W4mag',
 'Jmag',
 'Hmag',
 'Kmag',]]

,rImag,Hamag,imag,W1mag,W2mag,W3mag,W4mag,Jmag,Hmag,Kmag
126,16.5,14.64,15.64,11.726,11.021,8.111,5.91,13.864,13.031,12.502


# Noise

In [87]:
#df_noise_clean
# Clasificar todos los objetos en df_g4_clean
df_noise_clean['syst_class'] = classify_all_syst(df_noise_clean)

# Ver distribución de clases
print("Distribución de clasificaciones SySt:")
print(df_noise_clean['syst_class'].value_counts())

# Filtrar solo objetos clasificados como SySt
syst_objects_noise = df_noise_clean[df_noise_clean['syst_class'] != 'Not SySt']
print(f"\nTotal de objetos clasificados como SySt: {len(syst_objects_noise)}")

# Mostrar detalles de los SySt
print(syst_objects_noise[['J_H', 'Ks_W3', 'W1_W2', 'W1_W4', 'H_W2', 'W3_W4', 'syst_class']])

Distribución de clasificaciones SySt:
syst_class
Not SySt     944
S-type         3
S+IR-type      1
Name: count, dtype: int64

Total de objetos clasificados como SySt: 4
        J_H  Ks_W3  W1_W2  W1_W4   H_W2  W3_W4 syst_class
429   0.823  0.761 -0.036  1.849  0.482  1.296     S-type
976   1.269  1.042  0.052  1.295  0.866  0.512     S-type
996   1.278  0.819  0.011  1.185  0.870  0.696     S-type
1460  0.832  1.017  0.082  4.996  0.461  4.093  S+IR-type


/tmp/ipykernel_88192/1449914213.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_noise_clean['syst_class'] = classify_all_syst(df_noise_clean)
